In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [35]:
df = pd.read_csv('../datasets/DesmatamentoCidades.csv', sep=';')

df = df.rename(columns={
    'year': 'ano',
    'areakm': 'area_km2',
    'municipality': 'cidade',
})

print(df.columns)
df['area_km2'] = df['area_km2'].str.replace(',', '.')

# for _, row in df.iterrows():
#     print(row)


Index(['ano', 'area_km2', 'cidade', 'geocode_ibge', 'state'], dtype='object')


In [36]:
import unicodedata

def normalizar(texto):
    unaccent = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return unaccent.lower()

In [37]:
# Armazena dados prontos para inserir depois
dados_desmatamento = []

for _, row in df.iterrows():
    ano = row['ano']
    area_km2 = row['area_km2']
    cidade = row['cidade']
    cidade = normalizar(cidade)

    # Pega o id da area indígena
    cursor.execute("""
        SELECT id_cidade FROM cidade
        WHERE nome_normalizado = %s
    """, (cidade,))
    
    result = cursor.fetchone()
    id_cidade = result[0] if result else None

    if id_cidade is None:
        continue

    dados_desmatamento.append((ano, area_km2, id_cidade))

In [38]:
from psycopg2.extras import execute_values

# Agora, insere **em lote**:
query = """
    INSERT INTO relatorio_desmatamento
    (ano, area_km2, id_cidade)
    VALUES %s
"""
execute_values(cursor, query, dados_desmatamento)

# Finaliza
conn.commit()

In [ ]:
#Caso comando dê erro, desfaz as alterações
conn.rollback()

In [39]:
cursor.close()
conn.close()